In [1]:
import gzip
import ast
import pandas as pd

#load in the meta data
rows = []

with gzip.open('endomondoMeta.json.gz') as f:
    for line in f:
        row = ast.literal_eval(line.decode('utf-8'))
        
        #skip row if it doesnt contain weather data
        if row['weather'] is None:
            continue
            
        rows.append({
            'user_id': row['userId'],
            'sport': row['sport'],
            'gender': row['gender'],
            'weather': row['weather']['type'],
            'date': row['timestamp'][0],
            'duration': row['duration'],
            'distance': row['distance'],
            'calories': row['calories']
        })

df = pd.DataFrame(rows)
print(f"Total workouts loaded: {len(df)}")
print(df.head())

Total workouts loaded: 670266
    user_id          sport gender  weather                      date  \
0  10014612  mountain bike   male        7  2014-04-11T14:32:35.000Z   
1  10014612  mountain bike   male        7  2014-04-10T14:56:01.000Z   
2  10014612  mountain bike   male        6  2014-04-07T17:18:13.000Z   
3  10014612  mountain bike   male        1  2014-03-30T14:01:32.000Z   
4  10014612  mountain bike   male        1  2014-03-28T16:06:04.000Z   

   duration  distance  calories  
0   3218.35  18.85551    1313.0  
1   1728.33  10.91231     724.0  
2   1904.69  10.76309     424.0  
3   7218.46  38.31579    2732.0  
4   3208.09  18.71360    1301.0  


In [10]:
#Explore my data

#total number of workouts
print(f"Total Workouts: {len(df)}")

#how many users there are
print(f"Unique Users: {df['user_id'].nunique()}")

# what weather types are most common
print(f"Weather type counts: {df['weather'].value_counts().head(10)}")


#the differant sports in the data set
print(f"Sport Types: {df['sport'].value_counts().head(5)}")




# check for any missing values
print(f"Missing values:")
print(df.isnull().sum().head(5))

# check date range
print(f"\nDate range:")
print(f"Earliest: {df['date'].min()}")
print(f"Latest: {df['date'].max()}")




Total Workouts: 670266
Unique Users: 1361
Weather type counts: weather
1     142368
3     116921
6      80652
7      76626
4      42689
33     36819
2      31452
11     25460
35     21709
38     21415
Name: count, dtype: int64
Sport Types: sport
run                 244888
bike                180154
bike (transport)     80050
walk                 78530
mountain bike        29206
Name: count, dtype: int64
Missing values:
user_id    0
sport      0
gender     0
weather    0
date       0
dtype: int64

Date range:
Earliest: 1980-01-06T00:07:49.000Z
Latest: 2019-04-07T00:46:30.000Z

Workouts per user:
Average: 492.5
Min: 1
Max: 2309
Users with 20+ workouts: 1334


In [15]:
#clean up the data 

# convert date to proper datetime format
df['date'] = pd.to_datetime(df['date'])
print(f"Sample date: {df['date'].iloc[0]}")

#merge bike and bike transport into one category
df['sport'] = df['sport'].replace('bike (transport)', 'bike')

# filter to only keep sports we want
sports_to_keep = ['run', 'bike', 'walk']
df = df[df['sport'].isin(sports_to_keep)]

# remove users with less than 20 workouts per sport
workouts_per_user_sport = df.groupby(['user_id', 'sport']).size()
combos_to_keep = workouts_per_user_sport[workouts_per_user_sport >= 20].index
df = df[df.set_index(['user_id', 'sport']).index.isin(combos_to_keep)]



print("\nCleaned dataframe sample:")
print(df.head())

Sample date: 2013-07-24 19:43:38+00:00

Cleaned dataframe sample:
     user_id sport gender  weather                      date  duration  \
49  10014612  bike   male       12 2013-07-24 19:43:38+00:00   2853.39   
50  10014612  bike   male        1 2013-07-24 18:43:37+00:00   2434.31   
51  10014612  bike   male        1 2013-07-23 17:08:58+00:00   4267.55   
52  10014612  bike   male        1 2013-07-22 08:13:21+00:00   7414.72   
53  10014612  bike   male        1 2013-07-19 18:42:28+00:00   2129.50   

     distance  calories  
49  24.370959     801.0  
50   8.151629     769.0  
51  24.783918    1269.0  
52  34.269641    1575.0  
53  10.890956     399.0  
